# preprocessing calculated data

Sanitised source snapshot. See the repository README and WORKFLOW.md before running. Outputs and execution metadata are removed. Historical experiments are not an unattended pipeline or reproduced scientific result.


In [ ]:
import os
from pathlib import Path

REPO_ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "prepare_wells.py").is_file())
WORKSPACE_ROOT = Path(os.environ.get("CGWB_WORKSPACE", str(REPO_ROOT / "data" / "cgwb"))).expanduser().resolve()
WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE_ROOT)


In [ ]:
import pandas as pd

In [ ]:
import os
import pandas as pd

# Define the directory containing the CSV files
directory_path = f"{WORKSPACE_ROOT}/8. Creating aquifer wise training data/input/"

# List to store dataframes
dataframes = []

# Loop through each file in the directory
for filename in os.listdir(directory_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(directory_path, filename)
        df = pd.read_csv(file_path)
        dataframes.append((filename, df))

# Display each dataframe
for filename, df in dataframes:
    print(f"File: {filename}")
    display(df.head())  # Displaying the first few rows of each dataframe

In [ ]:
# Function to rename the columns based on the given criteria
def rename_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith('-01-01'):
            new_columns[col] = col[:4] + '_Jan_March'
        elif col.endswith('-04-01'):
            new_columns[col] = col[:4] + '_April_June'
        elif col.endswith('-07-01'):
            new_columns[col] = col[:4] + '_Jul_Sept'
        elif col.endswith('-10-01'):
            new_columns[col] = col[:4] + '_Oct_Dec'

    df.rename(columns=new_columns, inplace=True)

    return df

# Loop through each file in the directory and process them
modified_calculated_data = []
for filename in os.listdir(directory_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(directory_path, filename)
        df = pd.read_csv(file_path)
        df = rename_columns(df)
        modified_calculated_data.append((filename, df))

# Display the first few rows of each dataframe to check the changes
for filename, df in modified_calculated_data:
    print(f"File: {filename}")
    display(df.head())

In [ ]:
import json

In [ ]:
# Function to rename the columns based on the given criteria
def rename_columns(df):
    new_columns = {}
    for col in df.columns:
        if col.endswith('-01-01'):
            new_columns[col] = col[:4] + '_Jan_March'
        elif col.endswith('-04-01'):
            new_columns[col] = col[:4] + '_April_June'
        elif col.endswith('-07-01'):
            new_columns[col] = col[:4] + '_Jul_Sept'
        elif col.endswith('-10-01'):
            new_columns[col] = col[:4] + '_Oct_Dec'

    df.rename(columns=new_columns, inplace=True)
    return df

# Function to expand dictionary columns into separate columns
def expand_dict_columns(df):
    dict_columns = [col for col in df.columns if col != '.geo' and df[col].apply(lambda x: isinstance(x, str) and x.startswith('{')).any()]

    for col in dict_columns:
        dict_col_df = df[col].apply(lambda x: pd.Series(json.loads(x))).add_prefix(f'{col}_calculated_')
        df = df.drop(columns=[col]).join(dict_col_df)

    return df

# Loop through each file in the directory and process them
modified_calculated_data = []
for filename in os.listdir(directory_path):
    if filename.endswith(".csv"):
        file_path = os.path.join(directory_path, filename)
        df = pd.read_csv(file_path)
        df = rename_columns(df)
        df = expand_dict_columns(df)
        if 'system:index' in df.columns:
            df = df.drop(columns=['system:index'])
        modified_calculated_data.append((filename, df))

# Display the first few rows of each dataframe to check the changes
for filename, df in modified_calculated_data:
    print(f"File: {filename}")
    display(df.head())

In [ ]:
output_directory_path = f"{WORKSPACE_ROOT}/8. Preprocessing calculated WD data/output/"

In [ ]:
# Save the modified dataframes back to CSV files
for filename, df in modified_calculated_data:
    save_path = os.path.join(output_directory_path, f"preprocessed_{filename}")
    df.to_csv(save_path, index=False)